# Experimento principal

## Imports necesarios

In [2]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, hinge_loss
from joblib import Parallel, delayed

## Funciones 

In [3]:
def procesar_dataset(archivo):
    """
    Función objetivo para joblib. Procesa un único dataset completo usando SVC.
    Retorna el nombre del archivo, las métricas, las estadísticas y posibles errores.
    """
    if not os.path.exists(archivo):
        return archivo, None, None, f"[ERROR] Archivo no encontrado: {archivo}"

    df = pd.read_csv(archivo)
    X = df.drop(columns=['clase']).values
    y = df['clase'].values

    # Estructura ampliada para incluir las tres exactitudes
    metricas = {
        'acc_train': [], 
        'acc_val': [], 
        'acc_test': [], 
        'recall_test': [], 
        'f1_test': [], 
        'auc_test': []
    }
    
    sss_test = StratifiedShuffleSplit(n_splits=10, test_size=0.15, random_state=42)

    fold = 1
    for train_val_idx, test_idx in sss_test.split(X, y):
        X_train_val, X_test = X[train_val_idx], X[test_idx]
        y_train_val, y_test = y[train_val_idx], y[test_idx]

        sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.17647, random_state=42)
        for train_idx, val_idx in sss_val.split(X_train_val, y_train_val):
            X_train, X_val = X_train_val[train_idx], X_train_val[val_idx]
            y_train, y_val = y_train_val[train_idx], y_train_val[val_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)

        # Utilizando la implementación clásica de SVM analítica
        svm = SVC(kernel='linear', random_state=42)
        svm.fit(X_train, y_train)

        # Predicciones para calcular exactitud en los tres conjuntos
        y_pred_train = svm.predict(X_train)
        y_pred_val = svm.predict(X_val)
        y_pred_test = svm.predict(X_test)
        
        # Función de decisión requerida para calcular el AUC en SVC
        y_scores_test = svm.decision_function(X_test)

        # Almacenar métricas en el diccionario
        metricas['acc_train'].append(accuracy_score(y_train, y_pred_train))
        metricas['acc_val'].append(accuracy_score(y_val, y_pred_val))
        metricas['acc_test'].append(accuracy_score(y_test, y_pred_test))
        metricas['recall_test'].append(recall_score(y_test, y_pred_test))
        metricas['f1_test'].append(f1_score(y_test, y_pred_test))
        metricas['auc_test'].append(roc_auc_score(y_test, y_scores_test))

        fold += 1

    df_metricas = pd.DataFrame(metricas)
    df_metricas.index = [f"Fold {i+1}" for i in range(10)]

    stats = pd.DataFrame({
        'Media': df_metricas.mean(),
        'Mediana': df_metricas.median(),
        'Desv. Est.': df_metricas.std()
    }).T

    return archivo, df_metricas, stats, None

def entrenar_evaluar_svm_paralelo():
    archivos = [
        "radiomics_DE_rand_1.csv",
        "radiomics_DE_best_1.csv",
        "radiomics_DE_rand_2.csv",
        "radiomics_DE_best_2.csv"
    ]

    print(f"[INFO] Iniciando procesamiento en paralelo para {len(archivos)} datasets con joblib...")
    print(f"[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).")

    # Ejecución concurrente
    resultados = Parallel(n_jobs=-1)(delayed(procesar_dataset)(archivo) for archivo in archivos)

    # Impresión de resultados
    for archivo, df_metricas, stats, error in resultados:
        if error:
            print(error)
            continue

        print(f"\n{'-'*60}")
        print(f"[INFO] Resultados Dataset: {archivo}")
        print(f"{'-'*60}")
        print(f"\n[RESULTADOS] 10 Folds:")
        print(df_metricas.round(4).to_string())

        print(f"\n[ESTADISTICAS] Finales ({archivo}):")
        print(stats.round(4).to_string())
        print("\n")

In [4]:
if __name__ == "__main__":
    entrenar_evaluar_svm_paralelo()

[INFO] Iniciando procesamiento en paralelo para 4 datasets con joblib...
[INFO] Utilizando todos los núcleos disponibles (n_jobs=-1).

------------------------------------------------------------
[INFO] Resultados Dataset: radiomics_DE_rand_1.csv
------------------------------------------------------------

[RESULTADOS] 10 Folds:
         acc_train  acc_val  acc_test  recall_test  f1_test  auc_test
Fold 1      0.9290   0.9386    0.9352       0.9626   0.9558    0.9636
Fold 2      0.9314   0.9363    0.9374       0.9579   0.9571    0.9685
Fold 3      0.9329   0.9363    0.9352       0.9563   0.9556    0.9639
Fold 4      0.9373   0.9249    0.9226       0.9610   0.9477    0.9482
Fold 5      0.9368   0.9283    0.9238       0.9563   0.9482    0.9532
Fold 6      0.9300   0.9363    0.9386       0.9594   0.9579    0.9678
Fold 7      0.9348   0.9249    0.9272       0.9501   0.9501    0.9758
Fold 8      0.9375   0.9386    0.9113       0.9470   0.9396    0.9475
Fold 9      0.9344   0.9283    0.9272 